In [23]:
import numpy as np
import pandas as pd
import os, re, csv
from dotenv import load_dotenv

load_dotenv()

True

In [24]:
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk import word_tokenize, sent_tokenize
import string
from string import punctuation

In [25]:
def cleanData(text, stemming = False, lemmatize=False):    
    text = text.lower().split()
    text = " ".join(text)
    text = re.sub(r"[^A-Za-z0-9^,!.\/'+\-=]", " ", text)
    text = re.sub(r"what's", "what is ", text)
    text = re.sub(r"\'s", " ", text)
    text = re.sub(r"\'ve", " have ", text)
    text = re.sub(r"can't", "cannot ", text)
    text = re.sub(r"n't", " not ", text)
    text = re.sub(r"i'm", "i am ", text)
    text = re.sub(r"\'re", " are ", text)
    text = re.sub(r"\'d", " would ", text)
    text = re.sub(r"\'ll", " will ", text)
    text = re.sub(r",", " ", text)
    text = re.sub(r"\.", " ", text)
    text = re.sub(r"!", " ! ", text)
    text = re.sub(r"\/", " ", text)
    text = re.sub(r"\^", " ^ ", text)
    text = re.sub(r"\+", " + ", text)
    text = re.sub(r"\-", " - ", text)
    text = re.sub(r"\=", " = ", text)
    text = re.sub(r"'", " ", text)
    text = re.sub(r"(\d+)(k)", r"\g<1>000", text)
    text = re.sub(r":", " : ", text)
    text = re.sub(r" e g ", " eg ", text)
    text = re.sub(r" b g ", " bg ", text)
    text = re.sub(r" u s ", " american ", text)
    text = re.sub(r"\0s", "0", text)
    text = re.sub(r" 9 11 ", "911", text)
    text = re.sub(r"e - mail", "email", text)
    text = re.sub(r"j k", "jk", text)
    text = re.sub(r"\s{2,}", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    if stemming:
        st = PorterStemmer()
        txt = " ".join([st.stem(w) for w in text.split()])
    if lemmatize:
        wordnet_lemmatizer = WordNetLemmatizer()
        txt = " ".join([wordnet_lemmatizer.lemmatize(w) for w in text.split()])
    return text

In [26]:
special_character_removal=re.compile(r'[^a-z\d ]',re.IGNORECASE)
replace_numbers=re.compile(r'\d+',re.IGNORECASE)

def text_to_wordlist(text, remove_stopwords=False, stem_words=False):
    text = text.lower().split()
    
    text = " ".join(text)
    
    text=special_character_removal.sub('',text)
    text=replace_numbers.sub('',text)
    
    return(text)

In [27]:
train_df = pd.read_csv(os.getenv('train_csv'))
test_df = pd.read_csv(os.getenv('test_csv'))

train_df['comment_text'] = train_df['comment_text'].map(lambda x: cleanData(x))
test_df['comment_text'] = test_df['comment_text'].map(lambda x: cleanData(x))

In [34]:
classes = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

comments_train = train_df['comment_text'].values
comments_test = test_df['comment_text'].values

In [35]:
sentences_train=[]
for text in comments_train:
    sentences_train.append(text_to_wordlist(text))

sentences_test=[]
for text in comments_test:
    sentences_test.append(text_to_wordlist(text))

In [36]:
sentences_train[1]

'd aww  he matches this background colour i am seemingly stuck with thanks talk   january   utc'

In [33]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.layers import *
from keras.models import Model
from tensorflow.keras.layers import BatchNormalization

In [38]:
MAX_SEQUENCE_LENGTH = 150
MAX_NB_WORDS = 100000
EMBEDDING_DIM = 300
VALIDATION_SPLIT = 0.1

In [41]:
tokenizer =  Tokenizer(num_words=MAX_NB_WORDS)
tokenizer.fit_on_texts(sentences_train + sentences_test)

In [59]:
seq_train = tokenizer.texts_to_sequences(sentences_train)
seq_test = tokenizer.texts_to_sequences(sentences_test)

X = pad_sequences(
    seq_train,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    seq_test,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding='post',
    truncating='post'
)

y = train_df[classes]

In [60]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VALIDATION_SPLIT, random_state=68
)

In [56]:
import pickle

with open('tokenizer_r.pkl','wb') as f:
    pickle.dump(tokenizer,f)

In [61]:
print("Training Shape :", X_train.shape)
print("Validation Shape :", X_val.shape)
print("Test Shape :", X_test.shape)

print("Training Labels :", y_train.shape)
print("Validation Labels :", y_val.shape)

Training Shape : (143613, 150)
Validation Shape : (15958, 150)
Test Shape : (153164, 150)
Training Labels : (143613, 6)
Validation Labels : (15958, 6)


In [63]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout
from tensorflow.keras.layers import SimpleRNN, LSTM, Bidirectional

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint